In [1]:

#REVISION DE CALIDAD DE DATOS PARA CARGA EN POWER BI


>**IMPORTS**

In [2]:
from Configuracion import DOC_ANALISIS, DOC_RESULTADOS, DT_DATASET, DT_PROCESADOS, DT_VISTA

>**LIBRERIAS**

In [3]:

import pandas as pd
import numpy as np


>**LECTURA DE DATASET**

In [4]:
ruta_dt_operacion_sedes = DT_DATASET / 'operacion_sedes.csv'
df_operacion_sedes = pd.read_csv(ruta_dt_operacion_sedes)

print(f"DATASET CARGADO DE LA RUTA: {ruta_dt_operacion_sedes}")
df_operacion_sedes.head()


DATASET CARGADO DE LA RUTA: ..\Datos\Datasets\operacion_sedes.csv


,fecha,sede_id,region,generacion_kwh,demanda_kwh,evento_programado
0,2026-01-01,S01,Occidente,1564.65,1645.95,1
1,2026-01-01,S02,Occidente,1831.21,1714.67,1
2,2026-01-01,S03,Occidente,1998.01,1768.40,1
3,2026-01-01,S04,Centro,1723.11,1973.61,1
4,2026-01-01,S05,Centro,2064.75,2015.79,1


In [5]:
#Revision de datos

print(f"Filas: {df_operacion_sedes.shape[0]}")
print(f"Columnas: {df_operacion_sedes.shape[1]}")

print(f"Nombres Columnas: {df_operacion_sedes.columns.to_list()}")

print("\nTipado: ")
df_operacion_sedes.dtypes



Filas: 1620
Columnas: 6
Nombres Columnas: ['fecha', 'sede_id', 'region', 'generacion_kwh', 'demanda_kwh', 'evento_programado']

Tipado: 


fecha                    str
sede_id                  str
region                   str
generacion_kwh       float64
demanda_kwh          float64
evento_programado      int64
dtype: object

In [6]:
#Conteo de nulos

print("Valores nulos: ")

nulos = df_operacion_sedes.isnull().sum()
print(nulos)

print("\nValores Duplicados: ")
duplicados = df_operacion_sedes.duplicated().sum()
print(duplicados)

print("\nRevision de datos categoricos")
columnas_categoricas = ["region", "sede_id", "evento_programado"]
for columna in columnas_categoricas:
    print(f"\n{columna}")
    print(list(df_operacion_sedes[columna].unique()))

Valores nulos: 
fecha                0
sede_id              0
region               0
generacion_kwh       0
demanda_kwh          0
evento_programado    0
dtype: int64

Valores Duplicados: 
0

Revision de datos categoricos

region
['Occidente', 'Centro', 'Oriente']

sede_id
['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']

evento_programado
[np.int64(1), np.int64(0)]


>**TRANSFORMACION DE CAMPOS**

In [7]:
df_operacion_sedes['fecha'] = pd.to_datetime(df_operacion_sedes['fecha'], errors='coerce')

print("Fecha convertida: a formato: ", df_operacion_sedes['fecha'].dtypes)
print(df_operacion_sedes[['fecha']].head(5))

print("\nFechas nulas tras la conversión:", df_operacion_sedes['fecha'].isna().sum())

print("\nRango:", df_operacion_sedes['fecha'].min(), " - ", df_operacion_sedes['fecha'].max())

Fecha convertida: a formato:  datetime64[us]
       fecha
0 2026-01-01
1 2026-01-01
2 2026-01-01
3 2026-01-01
4 2026-01-01

Fechas nulas tras la conversión: 0

Rango: 2026-01-01 00:00:00  -  2026-06-29 00:00:00


>**DATOS NUMERICOS**

In [8]:
print("\nValores evento programado: ")
print(df_operacion_sedes['evento_programado'].unique())

print("\nRangos de campos flotantes")
print(df_operacion_sedes[['generacion_kwh', 'demanda_kwh']].describe())


Valores evento programado: 
[1 0]

Rangos de campos flotantes
       generacion_kwh  demanda_kwh
count     1620.000000  1620.000000
mean      1798.533056  1942.510290
std        324.636135   251.435598
min        991.390000  1294.290000
25%       1551.695000  1750.132500
50%       1787.495000  1941.690000
75%       2038.972500  2137.082500
max       2780.250000  2583.490000


In [9]:
#Revision de 

## Análisis de dataset

>En el analisis de calidad no se encontraron ninguna incositencia en los datos, se reviso nulos, duplicados, valores fuera del rngo esperado, columnas validas, categorias establecidas correctamente y datos numericos validos

### Transformaciones realizadas

>Se cambio el formato de fecha para ser compatible para una carga posterior en Power BI

>**CREACION DE VISTA DE INDICADORES**

In [10]:
#CLAVE VALOR MEDIDAS DAX

total_generada = df_operacion_sedes['generacion_kwh'].sum()

demanda_total = df_operacion_sedes['demanda_kwh'].sum()

diferencia = df_operacion_sedes['demanda_kwh'] - df_operacion_sedes['generacion_kwh']
deficit_energetico = diferencia[diferencia > 0].sum()

cobertura_porcentual = 100 * (1 - deficit_energetico /  df_operacion_sedes['demanda_kwh'].sum())


indicadores = pd.DataFrame({
    "indicador": ["energia_total_kwh", "demanda_total_kwh", "deficit_total_kwh", "cobertura_pct"],
    "valor": [round(total_generada, 2), round(demanda_total, 2), round(deficit_energetico, 2), round(cobertura_porcentual, 2)]
})

print(indicadores)

ruta_indicadores = DOC_RESULTADOS / 'indicadores.csv'
indicadores.to_csv(ruta_indicadores, index=False)


           indicador       valor
0  energia_total_kwh  2913623.55
1  demanda_total_kwh  3146866.67
2  deficit_total_kwh   286904.20
3      cobertura_pct       90.88


## INDICADORES

>Generacion total de energia kwh- Suma de todos los registros del campo

>Demanda Total kwh - Suma de todos los registros del campo

>Deficit energetico total - Suma de todos los registros calculando la resta entre la demanda y la generacion energetica, sin tomar en cuenta datos negativos relacionados a exdentes de la generacion

>Cobertura (%) - Total de demanda cubierta con la division del deficit entre la demanda sumada del entero, expresado en procetaje 



>**GUARDAR DATASET PARA CARGA EN POWER BI**

In [11]:

ruta_guardado = DT_PROCESADOS / 'dt_operacion_sedes_pb.csv'
df_operacion_sedes.to_csv(ruta_guardado, index=False, encoding='utf-8')

print(f"Dataset limpio guardado como: {ruta_guardado}")

Dataset limpio guardado como: ..\Datos\Procesados\dt_operacion_sedes_pb.csv


# Justificacion tecnica

**->Deficit_total_kwh calculado : 286904.20**

**->Cobertura_porcentual calculada: 90.88 %**

El déficit total no puede obtenerse a partir del déficit neto global, ya que los valores excedentes de la resta de  demanda total y producccion total  restarian al conjunto total de valores con deficits reales,para obtenerse se dejan los valores de sobreproduccion como 0 en cada registro y sumando los valores reales de deficits, el segmentador de region actua directamente sobre que cantidad de registros hara los calculos, y la cobertura se recalcula con el déficit y la demanda de esos registros, por eso no se puede obtener sumando o promediando coberturas de regiones.